# Phase 2 - Synthetic-to-Real Transfer Learning on MNIST

**Research question:** Can a CNN trained on synthetic digit images transfer to real handwritten MNIST digits, and which synthetic-data changes reduce the synthetic-to-real domain gap?

This notebook is the main graded notebook for Milestone 2. It runs the full experiment suite using the modular code in `src/synthetic_to_real_mnist/`.

## Rubric checklist

This notebook addresses Milestone 2 by:

1. Running a complete experiment set: real-data baseline, 10k vs 50k synthetic scale-up, improved synthetic generation, hybrid real+synthetic training, rotation/thickness ablations, and convolution-filter visualization.
2. Evaluating every trained model on the **same real MNIST test set**.
3. Saving concrete evidence: CSV result tables, training curves, confusion matrices, per-class accuracy plots, filter visualizations, and an automatically generated markdown report.
4. Interpreting results without claiming that synthetic data is better than real data unless the table supports it.

## 1. Setup

Run this notebook from the repository root. In Google Colab, the cell below clones the public GitHub repository if the source folder is not already present.

In [ ]:
# Colab / local setup
import os, sys
from pathlib import Path

REPO_URL = "https://github.com/isgandarjafarli/synthetic-to-real-mnist.git"

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and not Path("src/synthetic_to_real_mnist").exists():
    !git clone {REPO_URL}
    %cd synthetic-to-real-mnist

# Install only if needed. Colab usually already has torch/torchvision.
!pip -q install -r requirements.txt

REPO_ROOT = Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Repository root:", REPO_ROOT)

## 2. Imports and reproducibility

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from synthetic_to_real_mnist.config import SyntheticConfig, TrainConfig
from synthetic_to_real_mnist.synthetic_data import generate_synthetic_dataset
from synthetic_to_real_mnist.visualization import plot_sample_grid
from synthetic_to_real_mnist.experiments import build_phase2_specs, run_phase2_suite
from synthetic_to_real_mnist.reporting import write_phase2_report, update_readme_results
from synthetic_to_real_mnist.train import get_device, set_seed

set_seed(42)
device = get_device()
print("Device:", device)

## 3. Choose run mode

Use `RUN_MODE = "smoke"` only to verify that the notebook executes. Use `RUN_MODE = "full"` for the final submission because it runs the 50,000-sample synthetic experiments and the full Phase 2 comparison.

In [ ]:
RUN_MODE = "full"  # change to "smoke" only for debugging
OUTPUT_DIR = "outputs"
DATA_ROOT = "./data"
UPDATE_README_AFTER_RUN = True
SAVE_MODELS = False  # plots/results are enough for grading; checkpoints are large

base_train_cfg = TrainConfig(
    batch_size=128,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_workers=2 if device.type == "cuda" else 0,
    seed=42,
)

print(base_train_cfg)

## 4. Visual check: basic vs improved synthetic data

Before training, inspect whether the improved generator actually produces a wider range of digit styles. This supports the experimental motivation rather than treating synthetic data as a black box.

In [ ]:
basic_data, basic_labels = generate_synthetic_dataset(200, SyntheticConfig.basic(), seed=1, balanced=True, progress=False)
improved_data, improved_labels = generate_synthetic_dataset(200, SyntheticConfig.improved(), seed=2, balanced=True, progress=False)

plot_sample_grid(basic_data, basic_labels, "Basic synthetic samples", "outputs/basic_synthetic_samples.png")
plt.show()
plot_sample_grid(improved_data, improved_labels, "Improved synthetic samples", "outputs/improved_synthetic_samples.png")
plt.show()

## 5. Experiment plan

The table below is the exact suite that will run. All models use the same CNN architecture; the training data is the variable under study.

In [ ]:
specs = build_phase2_specs(RUN_MODE)
plan_rows = []
for s in specs:
    plan_rows.append({
        "experiment": s.experiment,
        "group": s.experiment_group,
        "train_source": s.train_source,
        "synthetic_samples": s.synthetic_samples,
        "real_fraction": s.real_fraction,
        "epochs": s.epochs,
        "generator": s.synth_config.name if s.synth_config else "MNIST real",
    })
plan_df = pd.DataFrame(plan_rows)
plan_df

## 6. Run Phase 2 experiments

This cell trains each model, evaluates it on real MNIST, and saves plots/tables under `outputs/`.

In [ ]:
results_df, trained_models, detailed = run_phase2_suite(
    mode=RUN_MODE,
    output_dir=OUTPUT_DIR,
    data_root=DATA_ROOT,
    base_train_cfg=base_train_cfg,
    save_model=SAVE_MODELS,
)

results_df

## 7. Main result table

Sort by real MNIST test accuracy. The real baseline is an upper-bound comparison, not the main claim; the key question is how close synthetic and hybrid training can get.

In [ ]:
results_df = pd.read_csv(f"{OUTPUT_DIR}/phase2_results.csv")
results_df.sort_values("real_test_accuracy", ascending=False)

## 8. Automatically generated interpretation

The next cell writes `outputs/phase2_report.md` and, if enabled, updates the marked results section in `README.md`. This prevents the README from containing fabricated or stale numbers.

In [ ]:
report_text = write_phase2_report(results_df, Path(OUTPUT_DIR) / "phase2_report.md")
print(report_text)

if UPDATE_README_AFTER_RUN:
    update_readme_results("README.md", results_df)
    print("README.md results section updated from actual experiment outputs.")

## 9. Saved figures

The runner saved training histories, confusion matrices, per-class accuracy plots, an experiment-summary bar chart, and convolution-filter visualizations. Use these figures in your explanation/defense.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

plot_dir = Path(OUTPUT_DIR) / "plots"
for file in [
    plot_dir / "phase2_experiment_summary.png",
    plot_dir / "conv1_filters_real_baseline.png",
    plot_dir / "conv1_filters_synthetic_improved.png",
]:
    if file.exists():
        print(file)
        display(Image(filename=str(file)))

## 10. Final reproducibility check

For Blackboard submission, submit the GitHub link and an updated ZIP of the same repository. The important generated files are listed below.

In [ ]:
important = [
    Path(OUTPUT_DIR) / "phase2_results.csv",
    Path(OUTPUT_DIR) / "phase2_report.md",
    Path(OUTPUT_DIR) / "plots" / "phase2_experiment_summary.png",
]
for p in important:
    print("FOUND" if p.exists() else "MISSING", p)

print("
Result files:")
for p in sorted(Path(OUTPUT_DIR).glob("**/*")):
    if p.is_file() and p.suffix.lower() in {".csv", ".md", ".png", ".json"}:
        print(p)